In [60]:
import os
import rootutils

from tqdm.notebook import tqdm

rootutils.setup_root(os.path.abspath('./'), indicator=".project-root", pythonpath=True, dotenv=True, cwd=True)

# auto-loading of imports from outside scripts
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [61]:
import pandas as pd

In [62]:
df_coord_numbs = pd.read_csv("data_cod/coord_numbs.csv")
df_coord_numbs.rename(columns={"smiles": "can_smiles"}, inplace=True)

In [63]:
# Drop rows and columns that contain only zeros:

rows_only_zeros = df_coord_numbs[(df_coord_numbs == 0).all(axis=1)]
columns_only_zeros = df_coord_numbs.loc[:, (df_coord_numbs == 0).all(axis=0)]

df_coord_numbs = df_coord_numbs.drop(columns=columns_only_zeros.columns, index=rows_only_zeros.index)

In [64]:
df_merged_temp = pd.read_csv("data_cod/cod_bradley_merged.csv")
df_merged_temp = df_merged_temp[df_merged_temp["id"].isin(df_coord_numbs["id"])]

# Only bradley, to use part of it for test:
df_bradley_temp = pd.read_csv("data_cod/bradley_with_cif.csv")
df_bradley_temp = df_bradley_temp[df_bradley_temp["id"].isin(df_coord_numbs["id"])]

---
## Creating Train and Test splits:
- 0.2 of Bradley is test, rest is train

In [65]:
TEST_BRADLEY_FRAC = 0.2

In [66]:
df_test = df_bradley_temp.sample(frac=TEST_BRADLEY_FRAC, random_state=42)

df_train = df_merged_temp.drop(index=df_merged_temp[df_merged_temp["id"].isin(df_test["id"])].index)

In [67]:
df_train = pd.merge(df_train, df_coord_numbs, on=["id", "can_smiles"], how="inner")
df_test = pd.merge(df_test, df_coord_numbs, on=["id", "can_smiles"], how="inner")

In [68]:
df_train

,id,can_smiles,T,cif_path,Ag–Ag,Al–H,H–Al,As–As,B–B,B–H,...,Se–Se,Si–H,H–Si,Si–Si,Sn–H,H–Sn,Sr–Sr,Ti–Ti,Zn–H,H–Zn
0,1000018,CC1(C)[C@@H]2CC[C@]3(C2)[C@H](O)CC[C@@H](O)[C@...,180.0,cifs/1000018.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1000019,CN(C)c1ccc(/C=C/C(=O)c2ccccc2O)cc1,-97.0,cifs/1000019.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1008189,OO,-40.0,cifs/1008189.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1008775,NC(N)=O,134.0,cifs/1008775.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1008776,NC(N)=O,134.0,cifs/1008776.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11852,9014479,ClCl,-101.0,cifs/9014479.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11853,9015081,c1ccc2c(c1)Cc1ccccc1-2,116.0,cifs/9015081.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11854,9015108,Cl,-114.2,cifs/9015108.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11855,9015583,ClCl,-101.0,cifs/9015583.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [69]:
df_train

,id,can_smiles,T,cif_path,Ag–Ag,Al–H,H–Al,As–As,B–B,B–H,...,Se–Se,Si–H,H–Si,Si–Si,Sn–H,H–Sn,Sr–Sr,Ti–Ti,Zn–H,H–Zn
0,1000018,CC1(C)[C@@H]2CC[C@]3(C2)[C@H](O)CC[C@@H](O)[C@...,180.0,cifs/1000018.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1000019,CN(C)c1ccc(/C=C/C(=O)c2ccccc2O)cc1,-97.0,cifs/1000019.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1008189,OO,-40.0,cifs/1008189.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1008775,NC(N)=O,134.0,cifs/1008775.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1008776,NC(N)=O,134.0,cifs/1008776.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11852,9014479,ClCl,-101.0,cifs/9014479.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11853,9015081,c1ccc2c(c1)Cc1ccccc1-2,116.0,cifs/9015081.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11854,9015108,Cl,-114.2,cifs/9015108.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11855,9015583,ClCl,-101.0,cifs/9015583.cif,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [70]:
X_train = df_train.drop(columns=["id", "can_smiles", "T", "cif_path"])
y_train = df_train["T"].astype(float)

X_test = df_test.drop(columns=["id", "can_smiles", "T"])
y_test = df_test["T"].astype(float)

---
## Training Catboost:

In [71]:
from catboost import CatBoostRegressor
from src.utils import eval_metrics

In [72]:
catboost_model = CatBoostRegressor(
    iterations=5000,
    # learning_rate=0.01,
    # depth=10,
    verbose=100
)
catboost_model.fit(X_train, y_train)

Learning rate set to 0.016353
0:	learn: 111.3327452	total: 5.93ms	remaining: 29.7s
100:	learn: 105.0372616	total: 260ms	remaining: 12.6s
200:	learn: 103.3742119	total: 512ms	remaining: 12.2s
300:	learn: 102.4177442	total: 761ms	remaining: 11.9s
400:	learn: 101.6992784	total: 1.01s	remaining: 11.6s
500:	learn: 101.1218911	total: 1.27s	remaining: 11.4s
600:	learn: 100.6153742	total: 1.53s	remaining: 11.2s
700:	learn: 100.2651447	total: 1.77s	remaining: 10.9s
800:	learn: 99.7896733	total: 2.02s	remaining: 10.6s
900:	learn: 99.1555672	total: 2.27s	remaining: 10.3s
1000:	learn: 98.5563563	total: 2.52s	remaining: 10.1s
1100:	learn: 98.0294329	total: 2.78s	remaining: 9.86s
1200:	learn: 97.5882323	total: 3.04s	remaining: 9.62s
1300:	learn: 97.1714254	total: 3.31s	remaining: 9.4s
1400:	learn: 96.7661885	total: 3.56s	remaining: 9.13s
1500:	learn: 96.3797691	total: 3.81s	remaining: 8.88s
1600:	learn: 96.0123621	total: 4.06s	remaining: 8.63s
1700:	learn: 95.6030991	total: 4.32s	remaining: 8.38s
18

In [73]:
y_pred = catboost_model.predict(X_test)

In [74]:
eval_metrics(y_test, y_pred, type="regression")

{'MSE': 3359.710538266327,
 'RMSE': 57.96301008631563,
 'MAE': 43.07557669717452,
 'R2': 0.5657370429640514}